## Concept 1 — Simple RAG (Manual Pipeline)

### What is it?
The most basic RAG. Every step is written manually so you can see exactly what happens at each stage.

### Pipeline
```
Query → Embed → Similarity Search → Top K Chunks → Prompt + LLM → Answer
```

### Limitation (what Concept 2 fixes)
Top K results may all say the same thing — no diversity. Pipeline is also fully manual.
→ Concept 2 introduces retriever abstraction and MMR for diverse results.

In [9]:
print("hi")

hi


In [10]:
import sys
sys.path.insert(0, '.')  # ensure RAGutils.py in same folder is importable

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()

Loading documents...
Splitting into chunks...
Creating embeddings and vector store...

Ready: 1 page(s) -> 11 chunks -> vector store created
LLM: gpt-4o-mini | Embeddings: text-embedding-3-small


### Step 1 — Manual Similarity Search
Directly query the vector store. Returns Top K most similar chunks by cosine similarity.

In [11]:
query = TEST_QUERIES[0]
print(f"Query: {query}\n")

# similarity_search returns a list of Document objects
results = vectorstore.similarity_search(query, k=3)

print(f"Retrieved {len(results)} chunks:")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:300])

Query: What is LangSmith persistence?

Retrieved 3 chunks:

--- Chunk 1 ---
Core resource data: assistants, threads, runs, and cron jobs. Always stored in PostgreSQL.
Checkpoints (short-term memory): snapshots of graph execution state written at each step. They make runs durable: if a worker is interrupted, the run can resume from the last checkpoint rather than from the be

--- Chunk 2 ---
Agent Server - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageDeploySearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAgent

--- Chunk 3 ---
Use Agent Server to create and manage:
AssistantsThreadsRunsCron jobs
API reference
For detailed information on the API endpoints and data models, refer to the Agent Server API reference.
​Application structure
To deploy an Agent Server application, you need to specify the graph(s) you w

### Step 2 — Build Context String
Join retrieved chunks into a single string to pass as context into the prompt.

In [12]:
context = format_docs(results)

print("Context passed to LLM (first 500 chars):")
print(context[:500])

Context passed to LLM (first 500 chars):
Core resource data: assistants, threads, runs, and cron jobs. Always stored in PostgreSQL.
Checkpoints (short-term memory): snapshots of graph execution state written at each step. They make runs durable: if a worker is interrupted, the run can resume from the last checkpoint rather than from the beginning. Durability mode controls checkpoint frequency—async (default) writes after each step; exit stores only the final state. LangSmith stores this in PostgreSQL by default; but you can switch to M


### Step 3 — Call LLM with Context + Question
Build the prompt with context and question, then invoke the LLM.

In [13]:
messages = prompt.format_messages(context=context, question=query)
response = llm.invoke(messages)

print(f"Q: {query}")
print(f"A: {response.content}")

Q: What is LangSmith persistence?
A: LangSmith persistence refers to the storage of core resource data, checkpoints, and long-term memory. Core resource data is stored in PostgreSQL by default, while checkpoints, which provide durability for runs, are also stored in PostgreSQL. Long-term memory, which allows agents to retain information across separate conversations, is similarly stored in PostgreSQL by default but can be replaced with a custom implementation.


### Full Function
Wrap all steps into one reusable function.

In [14]:
def simple_rag(query):
    results  = vectorstore.similarity_search(query, k=3)
    context  = format_docs(results)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries
Same 3 queries are used in every notebook so you can compare results across RAG techniques.

In [15]:
run_queries(simple_rag, label="Concept 1 — Simple RAG")


 Concept 1 — Simple RAG — Test Results

Q: What is LangSmith persistence?
A: LangSmith persistence involves storing core resource data in PostgreSQL, using checkpoints for durability during execution, and maintaining long-term memory through a Store that retains information across threads, also stored in PostgreSQL by default.
----------------------------------------

Q: What are checkpoints in LangSmith?
A: Checkpoints in LangSmith are snapshots of graph execution state written at each step. They make runs durable by allowing a run to resume from the last checkpoint if a worker is interrupted, rather than starting over from the beginning. The frequency of checkpoints can be controlled by durability mode, which has an async option that writes after each step and an exit option that stores only the final state.
----------------------------------------

Q: Explain LangSmith deployment
A: LangSmith cloud manages the database for you, simplifying the deployment process. If you choose to d